# Corporate bond pricing in OSEM

This notebook walks through how OSEM treats a corporate bond position: from raw input data, to projected cash flows, to a calibrated market-consistent price. Each corporate bond is defined by its characteristic, the market conditions such as spread and price. 

On a very high level:

1) Each corporate bond is saved into a CorpBond object.
2) A portfolio of CorpBond objects is then saved into the CorpBondPortfolio.
3) Based on the CorpBondPortfolio and market conditions, the cash flows resulting from each bond are reconstructed into a matrix
4) Based on the cash flows and current market conditions, the z-spread is calculated using the bisection algorithm to return the current market price 


## Table of contents

1. [Methodology](#section-1)
    - [Notation](#notation)
    - [Step 1 Cash flow dates](#step-1) · [Step 2 Date fractions](#step-2) · [Step 3 Cash flow amounts](#step-3)
    - [Step 4 Pricing off the risk-free curve](#step-4) · [Step 5 Calibrating the z-spread](#step-5)
    - [Step 6 Calibrating the whole portfolio](#step-6) · [Step 7 Use in the full OSEM run](#step-7)
    - [Generating new bonds](#new-bonds)
2. [Preparation before running the notebook](#section-2)
    - [Importing necessary external packages](#external-packages)
    - [Orient base folder](#base-folder)
    - [Import functions from the main code](#import-functions)
    - [Configuration and parameters](#configuration)
3. [Corporate bonds](#section-3)
4. [Projection of cash flows](#section-4)
    - [Importing the information about the economic environment](#economic-environment)
    - [Sector spread input](#sector-spread-input)
    - [Cash flow projection of a bond portfolio](#cash-flow-projection)
5. [Pricing and calibration](#section-5)
    - [Calculation of present value of each instrument](#present-value)
    - [Step 4 in numbers: the gap the z-spread has to close](#step-4-numbers)
    - [Calibrate the spread to match market price](#calibrate)
6. [Appendix: input data scheme](#section-6)

<a id="section-1"></a>

## 1. Methodology

How OSEM turns a bond description into a price, step by step. No code in this
section; the sections that follow run each step on the input portfolio.

<a id="notation"></a>

### Notation

This table defines the mathematical notation that will be used through the rest of the workbook

| Symbol | Meaning |
|---|---|
| $MD$ | modelling date |
| $VD$ | valuation date, the date cash flows are discounted to ($VD = MD$ at $t=0$) |
| $E$ | end of the modelling window (`settings.end_date`) |
| $I$ | issue date |
| $M$ | maturity date |
| $L$ | last cash flow date, $\min(E, M)$ |
| $N$ | notional amount |
| $c$ | coupon rate, paid on **each** coupon date |
| $f$ | coupon frequency (payments/year) |
| $MV$ | market value (price) at the modelling date |
| $z$ | bond-specific z-spread |
| $y(t)$ | risk-free spot yield for maturity $t$, from the calibrated EIOPA curve |

**A note on $f$.** The frequency sets only the *spacing* of the coupon dates, through
`relativedelta(months=12 // f)`. It does **not** scale the coupon amount: $c$ is the rate paid on
each coupon date, not an annualised rate, so a bond with $c = 0.02$ and quarterly frequency pays
2% of the notional four times a year. A bond quoted with an annual coupon $c_{ann}$ paid $f$ times
a year is entered in `Bond_Portfolio.csv` with `Coupon_Rate` $= c_{ann} / f$.

<a id="step-1"></a>

### Step 1 — Cash flow dates

Every bond produces two kinds of cash flow: coupons and the return of
notional.

A bond still alive when the modelling window closes is redeemed there at par,
so both kinds of cash flow stop at

$$ L = \min(E, M) $$

Notional repayment date:
$$ t_M^d = L $$

Coupon dates:
$$ I < t_1^d < t_2^d < \dots < t_k^d \leq L $$

The coupon grid is anchored on the issue date and stepped by $12/f$ months.
Payments falling before $MD$ are dropped, since only future cash flows are
priced.

<a id="step-2"></a>

### Step 2 — Date fractions relative to the valuation date

Only cash flows after the valuation date matter. Each remaining cash flow date
is converted to a year fraction from it:

$$ t_i = \frac{t_i^d - VD}{365.25} $$

At the modelling date $VD = MD$, which is the case this notebook demonstrates.
In a full run the valuation date moves forward with every period and the
fractions are re-measured from the new date.

This is what lets the discounting step treat every cash flow generically as
"an amount, this many years from now."

<a id="step-3"></a>

### Step 3 — Cash flow amounts

Notional cash flow (paid once, at maturity):
$$ cf_M = N $$

Coupon cash flows (paid at each coupon date):
$$ cf_i = N \cdot c $$

<a id="step-4"></a>

### Step 4 — Pricing off the risk-free curve

Before any bond-specific adjustment, each cash flow can be discounted purely
with the risk-free spot curve calibrated in `CurvesClass`:

$$ MV_{\text{rf}} = \sum_{i=1}^{k} \frac{cf_i}{(1+y(t_i))^{t_i}} + \frac{cf_M}{(1+y(t_M))^{t_M}} $$

This is *not* the bond's actual market price — it ignores credit/liquidity
risk entirely. The gap between it and the real `Market_Price` from the input
file is exactly what the z-spread in Step 5 is calibrated to close.

<a id="step-5"></a>

### Step 5 — Calibrating the z-spread to match the market price

OSEM finds a single constant $z$ added to the risk-free yield at every
maturity such that the discounted cash flows reproduce the bond's observed
market price exactly:

$$ MV = \sum_{i=1}^{k} \frac{cf_i}{(1+y(t_i)+z)^{t_i}} + \frac{cf_M}{(1+y(t_M)+z)^{t_M}} $$

There is no closed-form solution for $z$, so OSEM finds it numerically with a
**bisection search** (`bisection_spread`) over a bracket
$[z_{\text{start}}, z_{\text{end}}]$, which `calibrate_bond_portfolio` fixes at
$[-0.2, 0.2]$.

**The bracket has to contain a solution.** The price falls monotonically in $z$,
so a root exists only if $MV$ lies between the prices at the two ends of the
bracket. `bisection_spread` checks this before iterating and raises a
`ValueError` naming the bond when it fails. Without that check the search never
sees a sign change, walks $z_{\text{start}}$ all the way up to
$z_{\text{end}}$, and returns the bracket bound as though it were a calibrated
spread. A market price outside the bracket usually means either that the
bracket is too narrow for a genuinely distressed bond, or that `Coupon_Rate`
was entered as an annualised rate (see the note on $f$ above).

**What `precision` measures.** Once bracketed, the search stops when the
bracket is narrower than `precision`, that is
$(z_{\text{end}} - z_{\text{start}})/2 <$ `precision`. This is a tolerance on
the *spread*, not on the price. `calibrate_bond_portfolio` hardcodes
`precision` $= 10^{-8}$, which leaves a residual of order $10^{-6}$ in price on
the bonds in this portfolio. The two bracket endpoints are the exception: each
is returned straight away if it already prices within `precision` of $MV$.

<a id="step-6"></a>

### Step 6 — Calibrating the whole portfolio

`calibrate_bond_portfolio` simply repeats Step 5's bisection for every bond
in the portfolio, storing one $z$ per asset in `zspread_df`.


<a id="step-7"></a>

### Step 7 — How this is used in the full OSEM run

This notebook prices bonds once, at $t=0$. In a full run (`main.py`):

1. The portfolio's z-spreads are calibrated **once**, at the modelling date
   (`src/osem/main.py:253`).
2. In every subsequent period of the simulation, `price_bond_portfolio` is
   called again with that *same, fixed* z-spread but the *period's* point on
   the projected yield curve (`src/osem/main.py:324`) — so the bond's price moves only
   because the risk-free curve and remaining cash flows change, not because
   its credit spread is re-estimated.

This notebook only demonstrates the fixed-income leg in isolation; the full
run also handles equities, cash, liabilities/unit-linked policies, and
portfolio rebalancing each period.


<a id="new-bonds"></a>

### Generating new bonds (not yet implemented)

OSEM's documentation also describes a process for generating *new* corporate
bonds at each future period (to replace maturing debt), calibrating their
coupon so they price at par:

$$ 1 = \sum_{i=1}^{k} \frac{dy}{(1+y(t_i)+c+s+ss)^{t_i}} + \frac{1}{(1+y(t_M)+c+s+ss)^{t_M}} $$

This is not yet implemented in `BondClasses.py` / this notebook — see
`Documentation/OSEM_Documentation_draft.ipynb` (cells 84–99) for the full
methodology.

<a id="section-2"></a>

## 2. Preparation before running the notebook

<a id="external-packages"></a>

### Importing necessary external packages

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import datetime as dt
from pathlib import Path

<a id="base-folder"></a>

### Orient base folder

In [2]:
base_folder = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "ALM.ini").is_file())
sys.path.insert(0, str(base_folder / "src"))
os.chdir(base_folder)
base_folder = os.getcwd()  # Get current working directory

<a id="import-functions"></a>

### Import functions from the main code

In [3]:
from osem.CurvesClass import Curves
from osem.ImportData import import_SWEiopa, get_corporate_bonds, get_configuration, get_settings
from osem.BondClasses import *
from osem.ConfigurationClass import Configuration
from osem.MainLoop import create_cashflow_dataframe

<a id="configuration"></a>

### Configuration and parameters

In [4]:
conf: Configuration
conf = get_configuration(os.path.join(base_folder, "ALM.ini"), os)

These lines of code just extract the absolute location of different files:

In [5]:
parameters_file = conf.input_parameters
cash_portfolio_file = conf.input_cash_portfolio
bond_portfolio_file = conf.input_bond_portfolio

In [6]:
paramfile = pd.read_csv("Input/Parameters.csv").set_index("Parameter")

The parameter file is:

***

In [7]:
display(paramfile)

,Value
Parameter,
EIOPA_param_file,Input/Param_no_VA.csv
EIOPA_curves_file,Input/Curves_no_VA.csv
country,Slovenia
run_type,Risk Neutral
n_proj_years,50
Precision,1E-10
Tau,0.0001
compounding,-1
Modelling_Date,29/04/2023


***

The settings object holds data about file locations, information about the run settings and model parameters such as modelling date.

In [8]:
settings = get_settings(parameters_file)

The run that the rest of the notebook prices against:

***

In [9]:
run_settings = pd.DataFrame(
    # Rendered as strings: in a mixed-type column pandas applies one float format to every
    # numeric value, which would show a precision of 1e-10 as 0.0
    {"Value": [str(settings.modelling_date),
               str(settings.end_date),
               str(settings.n_proj_years),
               settings.country,
               f"{settings.precision:g}",
               f"{settings.tau:g}"]},
    index=["Modelling date",
           "End of modelling window",
           "Projection years",
           "Country",
           "Precision (curve calibration)",
           "Tau"],
)

display(run_settings)

,Value
Modelling date,2023-04-29
End of modelling window,2073-04-29
Projection years,50
Country,Slovenia
Precision (curve calibration),1e-10
Tau,0.0001


***

<a id="section-3"></a>

## 3. Corporate bonds

The CorpBond object contains information about each fixed income position. This includes:
* asset_id
* nace
* issuer
* issue_date
* maturity_date
* coupon_rate
* Comprehensive bond spread
* notional_amount
* frequency
* recovery_rate
* default_probability
* units
* market_price
  

A Python generator reads the bond portfolio file and encodes it into a dictionary based on the asset id. Each asset id contains a CorpBond object describing a single fixed income position.

In [10]:
bond_input_generator = get_corporate_bonds(bond_portfolio_file)
bond_input = {corp_bond.asset_id: corp_bond for corp_bond in bond_input_generator}

As an example, a single corporate bond in a CorpBond would look something like:

***

In [11]:
display(bond_input[1234])

CorpBond(asset_id=1234, nace='A1.4.5', issuer=None, issue_date=datetime.date(2021, 12, 3), maturity_date=datetime.date(2026, 12, 12), coupon_rate=0.03, notional_amount=100.0, spread_country=0.0, spread_sector=0.0, zspread=0.01, spread_stress=0.0, frequency=1, recovery_rate=0.4, default_probability=0.03, units=100.0, market_price=94.0)

***

The CorpBondPortfolio is just a dictionary of such objects.

In [12]:
bond_portfolio = CorpBondPortfolio(bond_input)

The whole fixed income portfolio at a glance. Note that `Annual coupon` is derived, not an
input: it is `Coupon rate` times `Frequency`, because the coupon rate is the rate paid on
each coupon date rather than an annualised one.

***

In [13]:
portfolio_overview = pd.DataFrame(
    [{"Maturity": bond.maturity_date,
      "Notional": bond.notional_amount,
      "Coupon rate": bond.coupon_rate,
      "Frequency": int(bond.frequency),
      "Annual coupon": bond.coupon_rate * int(bond.frequency),
      "Units": bond.units,
      "Market price": bond.market_price}
     for bond in bond_portfolio.corporate_bonds.values()],
    index=list(bond_portfolio.corporate_bonds),
)

display(portfolio_overview)

,Maturity,Notional,Coupon rate,Frequency,Annual coupon,Units,Market price
1234,2026-12-12,100.0,0.030000,1,0.03,100.0,94.0
2889,2028-12-12,100.0,0.025000,2,0.05,120.0,92.0
31,2025-12-03,100.0,0.003333,12,0.04,100.0,96.0
1,2030-06-30,100.0,0.005000,2,0.01,120.0,90.0
2,2035-12-31,100.0,0.010000,1,0.01,100.0,85.0
3,2026-12-12,100.0,0.030000,1,0.03,120.0,94.0
4,2028-12-12,100.0,0.025000,2,0.05,100.0,92.0
5,2025-12-03,100.0,0.003333,12,0.04,120.0,96.0
6,2026-12-12,100.0,0.030000,1,0.03,100.0,94.0
7,2028-12-12,100.0,0.025000,2,0.05,120.0,92.0


***

<a id="section-4"></a>

## 4. Projection of cash flows

<a id="economic-environment"></a>

### Importing the information about the economic environment

import_SWEiopa() reads the necessary data about the current yield curve. The risk free term structure it produces is what every bond cash flow is discounted against, before the bond specific z-spread is added on top. Inside OSEM, the parameters related to the yield curve are saved in the Curves object.

In [14]:
[maturities_country, curve_country, extra_param, Qb] = import_SWEiopa(settings.EIOPA_param_file,
                                                                          settings.EIOPA_curves_file, settings.country)
# Curves object with information about term structure
curves = Curves(extra_param["UFR"] / 100, settings.precision, settings.tau, settings.modelling_date,
                settings.country)

In [15]:
ufr = extra_param["UFR"]/100 # ultimate forward rate
precision = float(settings.precision) # Numeric precision of the optimisation
# Targeted distance between the extrapolated curve and the ufr at the convergence point
tau = float(settings.tau) # 1 basis point

In [16]:
curves.set_observed_term_structure(maturity_vec=curve_country.index.to_numpy(dtype=float), yield_vec=curve_country.values)
curves.calc_fwd_rates()
# The +1 matches main.py: the last projection period still has to be priced, so the
# projected curve has to reach one year past the end of the modelling window.
curves.project_forward_rate(settings.n_proj_years+1)
curves.calibrate_projected(settings.n_proj_years+1, 0.05, 0.5, 1000)

<a id="sector-spread"></a>

<a id="sector-spread-input"></a>

### Sector spread input

The sector spread file lists the NACE classification and the spread components that go with it. In this proof of concept the spread is taken directly from the bond input file, so `spreadfile` is loaded here for reference only.

In [17]:
spreadfile = pd.read_csv("Input/Sector_Spread.csv")
spreadfile.index = spreadfile["NACE"]
del spreadfile["NACE"]

<a id="cash-flow-projection"></a>

### Cash flow projection of a bond portfolio

The basis of OSEM is cash flow simulation. The cash flows for the coupon payment and the return of the notional are simulated separately. 

A list of dictionaries containing all the dates and amounts of coupon payments are produced by calling the create_coupon_flows function:

In [18]:
coupon_flows = bond_portfolio.create_coupon_flows(settings.modelling_date, settings.end_date)

The list of dictionaries containing the return of the notional amount is produced by calling the function create_maturity_flows:

In [19]:
notional_flows = bond_portfolio.create_maturity_flows(terminal_date=settings.end_date)

All cash flows can be represented in a matrix with all possible cash flow dates as columns and all equities as rows. The non-zero entries then represent the value of the cash flow at that date. The first step is to calculate the unique dates for the entire portfolio of bonds. This is done by calling the unique_dates_profiles() function over the dates related to coupons or notional amount payments.

Both can then conveniently be represented as DataFrames.

Note that a vector of bond specific spreads is also provided as output.

In [20]:
unique_list = bond_portfolio.unique_dates_profile(coupon_flows)

In [21]:
unique_terminal_list = bond_portfolio.unique_dates_profile(notional_flows)

Using the sorted list of unique dates as column headers, the dataframes containing the information related to the cash flows can be produced. 

The first dataframe contains the market price of each position. Additionally, the dataframe of zspreads is returned that helps to price the bonds using a discounted cash flow method. The last output is a dataframe containing the amount (units) of each bond in the portfolio is created. 

In [22]:
[market_price_df, zspread_df, units_df] = bond_portfolio.init_bond_portfolio_to_dataframe(settings.modelling_date)

A dataframe of cash flows and notional amount payments is created:

In [23]:
# Dataframe with bond  coupon cash flows
cash_flows = create_cashflow_dataframe(coupon_flows, unique_list)
# Dataframe with bond notional cash flows
notional_cash_flows = create_cashflow_dataframe(notional_flows, unique_terminal_list)

A compact view of the same two dictionaries, one row per bond. The last coupon never falls
after the date the notional is repaid, which is $L = \min(E, M)$ from Step 1:

***

In [24]:
coupon_schedule = pd.DataFrame({
    "Coupons": {aid: len(flows) for aid, flows in coupon_flows.items()},
    "Coupon amount": {aid: next(iter(flows.values())) for aid, flows in coupon_flows.items()},
    "First coupon": {aid: min(flows) for aid, flows in coupon_flows.items()},
    "Last coupon": {aid: max(flows) for aid, flows in coupon_flows.items()},
    "Notional repaid": {aid: min(flows) for aid, flows in notional_flows.items()},
})

display(coupon_schedule)

,Coupons,Coupon amount,First coupon,Last coupon,Notional repaid
1234,4,3.000000,2023-12-03,2026-12-03,2026-12-12
2889,12,2.500000,2023-06-03,2028-12-03,2028-12-12
31,32,0.333333,2023-05-03,2025-12-03,2025-12-03
1,15,0.500000,2023-06-30,2030-06-30,2030-06-30
2,13,1.000000,2023-12-31,2035-12-31,2035-12-31
3,4,3.000000,2023-12-03,2026-12-03,2026-12-12
4,12,2.500000,2023-06-03,2028-12-03,2028-12-12
5,32,0.333333,2023-05-03,2025-12-03,2025-12-03
6,4,3.000000,2023-12-03,2026-12-03,2026-12-12
7,12,2.500000,2023-06-03,2028-12-03,2028-12-12


***

Cash flow dataframe with coupon amounts and dates:

***

In [25]:
display(cash_flows.head())

,2023-05-03,2023-06-03,2023-06-30,2023-07-03,2023-08-03,2023-09-03,2023-10-03,2023-11-03,2023-12-03,2023-12-30,...,2029-06-30,2029-12-30,2029-12-31,2030-06-30,2030-12-31,2031-12-31,2032-12-31,2033-12-31,2034-12-31,2035-12-31
1234,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2889,0.000000,2.500000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,2.500000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31,0.333333,0.333333,0.0,0.333333,0.333333,0.333333,0.333333,0.333333,0.333333,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.000000,0.000000,0.5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.5,...,0.5,0.5,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0


***

Cash flow dataframe with notional amount payments and dates:

***

In [26]:
display(notional_cash_flows.head())

,2025-12-03,2026-12-12,2028-12-12,2030-06-30,2035-12-31
1234,0.0,100.0,0.0,0.0,0.0
2889,0.0,0.0,100.0,0.0,0.0
31,100.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,100.0,0.0
2,0.0,0.0,0.0,0.0,100.0


***

The extra spread due to the extra riskiness of the bond compared to a risk free instrument:

***

In [27]:
display(zspread_df)

,2023-04-29
1234,0.010
2889,0.010
31,0.010
1,0.005
2,0.005
3,0.010
4,0.010
5,0.010
6,0.010
7,0.010


***

<a id="section-5"></a>

## 5. Pricing and calibration

<a id="present-value"></a>

### Calculation of present value of each instrument

The cashflows can be used to price the current market value of the bond, implied by the assumed economic parameters.

At this point `zspread_df` still holds the `Z_Spread` column read from the portfolio file, so the prices below are discounted at the risk free curve **plus that input spread**, not at the risk free curve alone. They therefore do not reproduce the observed `Market_Price`; closing that gap is what the calibration in the next section does.

Note that `main.py` has no counterpart to this step. There the bond price at the modelling date is the `Market_Price` from the input file, and the first discounted cash flow price is produced inside the main loop.

For simplicity, this example does the pricing at the modelling date by setting the projection year equal to 0.

In [28]:
proj_period = 0

The present value of the bond implied by the current yield structure is:

In [29]:
market_price_df = bond_portfolio.price_bond_portfolio(cash_flows, notional_cash_flows, settings, proj_period, curves, zspread_df, market_price_df,settings.modelling_date)

***

In [30]:
display(market_price_df)

,2023-04-29
1234,97.618258
2889,107.485399
31,99.940951
1,85.547108
2,76.050676
3,97.618258
4,107.485399
5,99.940951
6,97.618258
7,107.485399


***

<a id="step-4-numbers"></a>

### Step 4 in numbers: the gap the z-spread has to close

Pricing the same cash flows at $z = 0$ gives $MV_{\text{rf}}$, the value of the bond if it carried
no credit or liquidity risk at all. Comparing the three columns shows what the calibration in the
next section is for: the risk free price is what the cash flows are worth on the EIOPA curve
alone, the market price is what the bond actually trades at, and the z-spread is the constant
addition to the curve that closes the difference.

***

In [31]:
risk_free_price = {}
for asset_id, bond in bond_portfolio.corporate_bonds.items():
    risk_free_price[asset_id] = bond.price_bond(cash_flows.loc[asset_id],
                                                notional_cash_flows.loc[asset_id],
                                                settings.modelling_date,
                                                proj_period,
                                                curves,
                                                0.0)

price_comparison = pd.DataFrame({
    "Risk free price": pd.Series(risk_free_price),
    "Price at input z-spread": market_price_df.iloc[:, 0],
    "Market price": pd.Series({aid: b.market_price
                               for aid, b in bond_portfolio.corporate_bonds.items()}),
})
price_comparison["Gap to close"] = (price_comparison["Risk free price"]
                                    - price_comparison["Market price"])

display(price_comparison)

,Risk free price,Price at input z-spread,Market price,Gap to close
1234,100.921641,97.618258,94.0,6.921641
2889,112.714855,107.485399,92.0,20.714855
31,102.345972,99.940951,96.0,6.345972
1,88.453864,85.547108,90.0,-1.546136
2,80.528539,76.050676,85.0,-4.471461
3,100.921641,97.618258,94.0,6.921641
4,112.714855,107.485399,92.0,20.714855
5,102.345972,99.940951,96.0,6.345972
6,100.921641,97.618258,94.0,6.921641
7,112.714855,107.485399,92.0,20.714855


***

<a id="calibrate"></a>

### Calibrate the spread to match market price

To calibrate the spread implied by the market, OSEM uses a bisection method to obtain the spread such that when added on top of the risk free term structure, the discounted cashflows equal to the current market price.

In [32]:
calibrated_spread = bond_portfolio.corporate_bonds[1234].bisection_spread(x_start=-0.2
                                , x_end=0.2
                                , modelling_date=settings. modelling_date
                                , end_date=settings.end_date
                                , proj_period=proj_period
                                , curves=curves
                                , precision= 0.00000001
                                , max_iter=100000)

The market value calculated using the discounted cash flow method using the calibrated zspread is:

In [33]:
calibrated_bond = bond_portfolio.corporate_bonds[1234].price_bond(cash_flows.loc[1234],notional_cash_flows.loc[1234],settings.modelling_date, proj_period,curves,calibrated_spread)

***

In [34]:
print(calibrated_bond)

94.00000089724024


***

The function to calibrate the entire portfolio:

In [35]:
zspread_df=bond_portfolio.calibrate_bond_portfolio(zspread_df, settings, proj_period, curves)

***

In [36]:
display(zspread_df)

,2023-04-29
1234,0.021480
2889,0.043776
31,0.027160
1,-0.002582
2,-0.004688
3,0.021480
4,0.043776
5,0.027160
6,0.021480
7,0.043776


***

Repricing the portfolio with the calibrated spreads reproduces the observed market prices.
The residual is the price cost of stopping the bisection once the spread bracket is
narrower than `precision`, as described in Step 5:

***

In [37]:
calibrated_price_df = bond_portfolio.price_bond_portfolio(cash_flows,
                                                         notional_cash_flows,
                                                         settings,
                                                         proj_period,
                                                         curves,
                                                         zspread_df,
                                                         market_price_df.copy(),
                                                         settings.modelling_date)

calibration_check = pd.DataFrame({
    "Calibrated z-spread": zspread_df.iloc[:, 0],
    "Repriced value": calibrated_price_df.iloc[:, 0],
    "Market price": pd.Series({aid: bond.market_price
                               for aid, bond in bond_portfolio.corporate_bonds.items()}),
})
calibration_check["Residual"] = (calibration_check["Repriced value"]
                                 - calibration_check["Market price"])

display(calibration_check)

,Calibrated z-spread,Repriced value,Market price,Residual
1234,0.021480,94.000001,94.0,8.972402e-07
2889,0.043776,92.000001,92.0,1.121744e-06
31,0.027160,96.000000,96.0,-3.093212e-07
1,-0.002582,90.000002,90.0,1.736422e-06
2,-0.004688,84.999995,85.0,-4.698248e-06
3,0.021480,94.000001,94.0,8.972402e-07
4,0.043776,92.000001,92.0,1.121744e-06
5,0.027160,96.000000,96.0,-3.093212e-07
6,0.021480,94.000001,94.0,8.972402e-07
7,0.043776,92.000001,92.0,1.121744e-06


***

<a id="section-6"></a>

## 6. Appendix: input data scheme

There are multiple input files needed to calibrate the fixed income portfolio. They are located in the "Input" folder.

<a id="parameters-csv"></a>

### Parameters.csv

Parameters file holds information about the type of run and the modelling date.

 - EIOPA_param_file ...the relative location of the EIOPA parameter file that will be used as the RFR Ex. "Input/Param_no_VA.csv"
 - EIOPA_curves_file ... the relative location of the EIOPA yield curve that will be used as the RFR Ex. "Input/Curves_no_VA.csv"
 - country ... the name of the country that will be used as the base for this run Ex. "Slovenia"
 - n_proj_years ... length of a run in years starting from the Modelling date Ex. 50
 - Precision ... precision parameter used when calibrating the risk free curve in the Curves object Ex. 1E-10. Note that the bond z-spread calibration does not use it: calibrate_bond_portfolio hardcodes a tolerance of 1e-8
 - Tau ... the acceptable size of the gap between the extrapolated yield rate and the ulitmate forward rate Ex. 0.0001
 - compounding ... the way that the interest rates are compounded in the run Ex. -1
 - Modelling_Date ... the starting date of the run specified as a date string Ex."29/04/2023"


<a id="eiopa-files"></a>

### EIOPA RFR files

There are two types of files derived from the monthly EIOPA RFR submission that are used in this model. The "Curves_XX.csv" containing the yearly yield curves for all countries in scope and the "Param_XX.csv" with the parameters used to derive the curves. These files are used to derive the risk free term structure at the modelling date and to efficiently project the evolution of the term structure.

<a id="portfolio-description"></a>

### Portfolio description

The modelled portfolio is split by asset classes. The fixed income portfolio is located in the file "Bond_Portfolio.csv". Each security needs the following fields:

 -  Asset ID ... unique id such as an ISIN, SEDOL or CUSIP code Ex. IT1234567891
 -  Asset_Type ... asset type string Ex. "Corporate_Bond"
 -  NACE ... NACE asset classification code (nomenclature statistique des activités économiques dans la Communauté européenne) Ex. A1.4.5
 -  Issue_Date ... the string date specifying the issue date of the bond Ex. 3/12/2021
 -  Maturity_Date ... the string date specifying the maturity date of the bond Ex. 3/12/2021
 -  Notional_amount ... the notional amount of the bond Ex. 100
 -  Coupon_Rate ... percentage of the notional amount paid in dividends every period (specified by Frequency) Ex. 0.0014
 -  Frequency ... number of times per a year that dividends are paid Ex. 1 (once per a year)
 -  Recovery_Rate ... percentage of the notional amount that can be recovered in case of a default Ex. 0.80
 -  Default_Probability ... percentage probability of default per year Ex. 0.012
 -  Units ... number of each bond held in the portfolio Ex. 230
 -  Market_Price ... market price of the bond at the modelling date Ex. 96

### Sector spread
The list of NACE sector codes and the sector specific spread over the risk free rate
 - NACE ... NACE code of the issuer Ex. "A1.1" 
 - NACE code text  ... description of the NACE code for this issuer Ex. "Growing of non-perennial crops" 
 - cSpread  ...  Country of issuance or operations specific spread Ex. 0.01
 - sSpread  ...  Sector specific spread Ex. 0.01
 - zSpread  ...  bond issuance sector specific spread Ex. 0.01
 - ssSpread  ... Extra spread assumed by the specific stress scenario Ex. 0.01

In the POC, the spread is displayed directly in the bond input file